# Caso con zona horaria declarada (P1-a, entrega 5a)
El mismo archivo, ingerido con la zona que el analista declara, en un **caso nuevo**. El caso original no se toca: documenta lo que se asumió entonces. Al final se comparan los hallazgos de ambos casos **por entidad**, no por identificador (el identificador incluye las métricas, y si alguna lleva una hora, cambiar la zona lo cambia aunque se detecte lo mismo).

In [ ]:
%load_ext autoreload
%autoreload 2
from dfir_copilot.profiling import inspect_source

RUTA = "/workspace/data/raw/three_months.csv"
IDIOMA = "es"
ZONA = "America/Santiago"      # nombre IANA exacto; una errata falla al instante con sugerencia
NOTA = "Declarada por el analista el 2026-10-05 (Chile por defecto); pendiente de confirmar con el dueño del export"

%time borrador = inspect_source(RUTA, lang=IDIOMA, timezone=ZONA, timezone_note=NOTA)
print(borrador.render())

## Revisa y guarda el borrador
La zona queda como `declared` y **sin verificar**. Cuando el dueño del export la confirme, cambia `timezone_verified` a `true` y anota en `timezone_note` quién y cuándo, en un caso nuevo (un caso sellado no cambia de mapping).

In [ ]:
RUTA_BORRADOR = "/workspace/data/mappings_borradores/three_months_santiago.yaml"
borrador.save(RUTA_BORRADOR)
print(open(RUTA_BORRADOR, encoding="utf-8").read().split("derived:")[0])   # solo la cabecera y la fecha

## Ingerir en un caso nuevo
Mira el bloque `timezone` del manifiesto: procedencia, nota y cuántas filas caen en horas que un cambio de horario repite o salta (con Chile en este periodo deben ser 0).

In [ ]:
from dfir_copilot.cases import CaseWorkspace

CASO = "IDOR-INVOICES-2020Q4-TZ-SANTIAGO"
ws = CaseWorkspace.open_or_create(CASO, root="/workspace/data/cases", analyst="eder", lang=IDIOMA)
ws.add_raw(RUTA, link=True)
if not ws.meta["dataset"]:
    manifest = ws.ingest(RUTA, RUTA_BORRADOR)
else:
    import json
    manifest = json.loads((ws.processed_dir / ws.meta["dataset"]["parquet"]).with_suffix(".manifest.json").read_text(encoding="utf-8"))
print(manifest["output"]["rows"], "filas | rango UTC:", manifest["time_range_utc"])
print("zona:", manifest["timezone"])
for w in manifest["warnings"]:
    print("aviso:", w)

## Detectores sobre el caso nuevo

In [ ]:
from dfir_copilot.tools.toolkit import Toolkit

engine = ws.engine()
ledger = ws.ledger(engine)
salida = Toolkit(engine, ledger).call("run_detectors")
print("detectores:", [(d["name"], d["status"], d["findings"]) for d in salida.data["detectors"]])
print("candidatos:", len(salida.data["candidates"]))

## Comparación con el caso original (solo lectura)
* **Mismas entidades**: se detectó a los mismos actores y recursos con cada detector.
* **Métricas distintas en entidades comunes**: lo esperable si una métrica lleva una hora o un corte temporal; la zona la desplaza 3 h.
* **Entidades solo en un caso**: lo importante. Significaría que la zona cambia *qué* se detecta, no solo *cuándo*.

In [ ]:
import json
from pathlib import Path

ORIGINAL_LEDGER = Path("/workspace/data/ledger/IDOR-INVOICES-2020Q4.jsonl")

def por_entidad(entradas):
    return {(e["detector"], json.dumps(e["entity"], sort_keys=True)): e for e in entradas}

if ORIGINAL_LEDGER.exists():
    original = por_entidad(json.loads(l)["data"] for l in ORIGINAL_LEDGER.read_text(encoding="utf-8").splitlines()
                           if l.strip() and json.loads(l)["type"] == "finding")
    nuevo = por_entidad(e["data"] for e in ledger.entries("finding"))
    comunes = original.keys() & nuevo.keys()
    distintas = sorted(k for k in comunes if original[k]["metrics"] != nuevo[k]["metrics"])
    print("hallazgos original:", len(original), "| este caso:", len(nuevo), "| mismas entidades:", len(comunes),
          "| solo aquí:", len(nuevo.keys() - original.keys()), "| solo en el original:", len(original.keys() - nuevo.keys()))
    print("entidades comunes con métricas distintas:", len(distintas))
    for detector, _ in distintas:
        print("  ", detector, "-> métricas que cambian:", sorted(m for m in original[(detector, _)]["metrics"]
                                                          if original[(detector, _)]["metrics"][m] != nuevo[(detector, _)]["metrics"].get(m)))
else:
    print("no hay ledger original en", ORIGINAL_LEDGER)

## Cadena de custodia

In [ ]:
print(ws.verify().render(IDIOMA))
print(ledger.summary()["counts"])